# Predicción de la impuntualidad en una ruta de transporte público de Cusco
## Notebook de experimentación — Entregables 2 y 3

**Curso:** 1INF02 Aprendizaje de Máquina — 2026-2 (PUCP)
**Responsable:** Jafeth Manases Cuyo Alata
**Equipo:** [completar]

---

### Cómo usar esta plantilla

Esta plantilla sigue el **diseño experimental del Entregable 1** (Sección V del paper). Está organizada por etapas, y cada etapa indica qué objetivo específico (OE) o hipótesis (H) cubre.

- Las celdas **ya implementadas** son solo de infraestructura: entorno, carga de datos y constantes del diseño.
- Las celdas marcadas con `# TODO` son las que el equipo debe implementar. Cada una incluye qué hacer y **por qué**, para que cada decisión pueda justificarse en el paper.
- Las celdas `✅ Verificación` comparan sus resultados con los valores obtenidos en el análisis exploratorio del Entregable 1. Si un valor no coincide, revisen su implementación **antes** de seguir.

| Etapa | Contenido | OE / H |
|---|---|---|
| 0–2 | Entorno, carga de datos y constantes | — |
| 3 | Preprocesamiento e integración | OE1 |
| 4 | Análisis exploratorio | OE2 |
| 5 | Ingeniería de variables (F0, F1, F2) | OE4 |
| 6 | Partición temporal | — |
| 7 | Líneas base | H1 |
| 8 | Modelos y ajuste de hiperparámetros | OE3, H1 |
| 9 | Evaluación en prueba y pruebas estadísticas | OE3, H1 |
| 10 | Ablaciones: efecto de k y del GPS | OE4, H2, H3 |
| 11 | Interpretación | OE5 |
| 12 | Reproducibilidad y exportación | — |

## 0. Entorno

Google Colab ya trae `pandas`, `numpy`, `scikit-learn`, `xgboost`, `scipy` y `matplotlib`. Se fija la semilla para que los resultados sean reproducibles (requisito de la tarea).

In [ ]:
# Descomentar solo si falta alguna librería
# !pip install -q xgboost

import os, sys, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn, xgboost, scipy
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             f1_score, recall_score, confusion_matrix,
                             average_precision_score, classification_report)
from sklearn.inspection import permutation_importance
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Python      ", sys.version.split()[0])
print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("scikit-learn", sklearn.__version__)
print("xgboost     ", xgboost.__version__)
print("scipy       ", scipy.__version__)

## 1. Carga de datos

`Monitoring.csv` pesa unos 120 MB. Subirlo con `files.upload()` en cada sesión es lento, así que se recomienda la **opción A**: guardar ambos archivos una vez en Google Drive y montarlo.

- **Opción A (recomendada):** crear en Drive la carpeta `AM2026-2/data/`, colocar `Times.csv` y `Monitoring.csv`, y ajustar `DRIVE_DIR` si usan otra ruta.
- **Opción B:** subir los archivos manualmente en cada sesión.

In [ ]:
MODO_CARGA = "drive"   # "drive" o "upload"
DRIVE_DIR = "/content/drive/MyDrive/AM2026-2/data"

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB and MODO_CARGA == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = DRIVE_DIR
elif EN_COLAB and MODO_CARGA == "upload":
    from google.colab import files
    print("Seleccione Times.csv y Monitoring.csv")
    files.upload()          # se guardan en /content
    DATA_DIR = "/content"
else:
    DATA_DIR = os.environ.get("DATA_DIR", "./data")   # ejecución local

PATH_TIMES = os.path.join(DATA_DIR, "Times.csv")
PATH_MON   = os.path.join(DATA_DIR, "Monitoring.csv")
for p in (PATH_TIMES, PATH_MON):
    print(("OK  " if os.path.exists(p) else "FALTA ") + p)

In [ ]:
# vehicle_id_id se lee como texto para conservar los ceros a la izquierda ("05")
times_raw = pd.read_csv(PATH_TIMES, dtype={"vehicle_id_id": str})
mon_raw   = pd.read_csv(PATH_MON,   dtype={"vehicle_id_id": str})

print("Times      :", times_raw.shape)
print("Monitoring :", mon_raw.shape)
display(times_raw.head(3))
display(mon_raw.head(3))

### ✅ Verificación de la carga

In [ ]:
assert times_raw.shape == (4707, 8),    f"Times: se esperaba (4707, 8), se obtuvo {times_raw.shape}"
assert mon_raw.shape   == (1630431, 9), f"Monitoring: se esperaba (1630431, 9), se obtuvo {mon_raw.shape}"
assert times_raw.vehicle_id_id.nunique() == 45 and mon_raw.vehicle_id_id.nunique() == 45
print("Carga correcta ✔")

## 2. Constantes del diseño experimental

Estos valores **son decisiones del Entregable 1** y deben mantenerse fijos para que los experimentos sean coherentes con el paper. Si el equipo decide cambiar alguno, debe justificarse y actualizarse también en el documento.

In [ ]:
N_PC = 18                     # puntos de control por vuelta
K_VALUES = [0, 4, 8, 12]      # PC conocidos al momento de predecir (0 = antes de la salida)
MAX_ABS_DEV = 60              # |desviación| > 60 min se trata como atípico (faltante)

# Clases de puntualidad (Sección V-A). Con la desviación truncada a minutos:
#   adelantado: y < -1   (adelanto de al menos 120 s)
#   puntual   : -1 <= y <= 5
#   retrasado : y > 5    (retraso de al menos 360 s)
UMBRAL_ADELANTO = -1
UMBRAL_RETRASO  = 5
CLASES = {0: "adelantado", 1: "puntual", 2: "retrasado"}

# Partición temporal (Sección V-D)
TEST_INICIO = "2025-01-28"    # prueba: 28/01 – 01/02 (5 días). Se usa UNA sola vez.
# Desarrollo: 06/01 – 27/01 (22 días), validación cruzada de ventana expansiva:
CV_FOLDS = [
    # (fin_entrenamiento, inicio_validacion, fin_validacion)
    ("2025-01-15", "2025-01-16", "2025-01-18"),
    ("2025-01-18", "2025-01-19", "2025-01-21"),
    ("2025-01-21", "2025-01-22", "2025-01-24"),
    ("2025-01-24", "2025-01-25", "2025-01-27"),
]
DEV_INICIO = "2025-01-06"

# Espacios de búsqueda de hiperparámetros (Sección V-D)
GRIDS_REG = {
    "ridge": {"alpha": [0.01, 0.1, 1, 10, 100]},
    "rf":    {"n_estimators": [200, 500], "max_depth": [None, 10, 20], "min_samples_leaf": [1, 5, 10]},
    "xgb":   {"n_estimators": [200, 400, 800], "learning_rate": [0.03, 0.1],
              "max_depth": [3, 4, 6], "subsample": [0.8, 1.0]},
}
GRIDS_CLF = {
    "logistic": {"C": [0.01, 0.1, 1, 10, 100], "class_weight": [None, "balanced"]},
    "rf":       {"n_estimators": [200, 500], "max_depth": [None, 10, 20], "min_samples_leaf": [1, 5, 10],
                 "class_weight": [None, "balanced"]},
    "xgb":      {"n_estimators": [200, 400, 800], "learning_rate": [0.03, 0.1],
                 "max_depth": [3, 4, 6], "subsample": [0.8, 1.0]},
    # XGBoost no tiene class_weight: para balancear se usa sample_weight en .fit()
}
ALPHA_ESTADISTICO = 0.05

## 3. Preprocesamiento e integración (OE1)

### 3.1 Transformar las listas de `Times.csv`

Las columnas `start_time`, `end_time` y `difference_time` son **cadenas con 18 valores** con el formato `{05:45:00,05:47:00,...}`.

Según la verificación del Entregable 1 (Sección IV-B):
- `start_time` contiene las horas **programadas** y `end_time` las horas **reales**.
- `difference_time` = `trunc((real − programada) / 60)` en minutos. Negativo significa adelanto.
- `"-"` en `difference_time` indica un valor **faltante**: la hora real está corrupta.

**Objetivo:** obtener tres matrices de forma `(n_vueltas, 18)`:
- `S`: horas programadas en segundos desde la medianoche.
- `E`: horas reales en segundos.
- `D`: desviaciones en minutos (`float`, con `NaN` en los faltantes).

**Por qué en segundos:** permite restar horas y comparar con los registros GPS, que también se convertirán a segundos.

In [ ]:
# TODO: implementar
def parse_lista(celda: str) -> list:
    # Convierte '{a,b,c}' en ['a', 'b', 'c']
    ...

def hora_a_segundos(hhmmss: str) -> int:
    # '05:45:00' -> 20700
    ...

# S = ...   # (n, 18) horas programadas en segundos
# E = ...   # (n, 18) horas reales en segundos
# D = ...   # (n, 18) desviaciones en minutos, NaN donde había "-"

#### ✅ Verificación 3.1

In [ ]:
def verificar_3_1(S, E, D):
    assert S.shape == E.shape == D.shape == (4707, 18)
    assert int(np.isnan(D).sum()) == 7746, "Debe haber 7 746 desviaciones '-' (9,14 % de 84 726)"
    valido = ~np.isnan(D)
    calc = np.trunc((E - S) / 60)
    assert (calc[valido] == D[valido]).all(), "difference_time debe ser trunc((real - prog)/60)"
    print("3.1 correcto ✔  (la fórmula (1) del paper se cumple en los 76 980 valores válidos)")

# verificar_3_1(S, E, D)

### 3.2 Valores atípicos y vueltas corruptas

1. **Atípicos:** marcar como `NaN` las desviaciones con `|d| > MAX_ABS_DEV`. Son 156 valores. **Por qué:** un retraso de horas no es un retraso operativo; indica un error de registro.
2. **Horario no monótono:** excluir las vueltas cuyas horas programadas **no son crecientes** de PC1 a PC18. Son 148 vueltas. **Por qué:** un itinerario desordenado es un registro corrupto, y además genera horarios ficticios.
3. Construir el DataFrame de vueltas `vueltas` con `id`, `date`, `lap`, `vehicle_id_id`, la hora programada de salida (`S[:,0]`) y la duración programada (`S[:,17] − S[:,0]`, en minutos).

In [ ]:
# TODO: implementar
# D[np.abs(D) > MAX_ABS_DEV] = np.nan
# monotono = ...
# vueltas = ...   # DataFrame con una fila por vuelta válida (y S, E, D filtradas igual)

#### ✅ Verificación 3.2

In [ ]:
def verificar_3_2(vueltas, S):
    assert len(vueltas) == 4559, f"Se esperaban 4 559 vueltas tras excluir 148, hay {len(vueltas)}"
    dur = (S[:, 17] - S[:, 0]) / 60
    assert sorted(set(dur)) == [147.0, 161.0, 168.0, 170.0], "Deben quedar solo 4 horarios programados"
    print("3.2 correcto ✔")

# verificar_3_2(vueltas, S)

### 3.3 Variable objetivo y clases

- **Regresión:** `y = D[:, 17]`, la desviación en PC18. Hay **una sola variable objetivo por vuelta**, porque cada fila es una vuelta completa de ida y vuelta (PC1 → PC10 → PC18).
- **Clasificación:** tres clases con `UMBRAL_ADELANTO` y `UMBRAL_RETRASO`.
- Excluir del modelado las vueltas sin `y` válido (555).

In [ ]:
# TODO: implementar
def a_clase(y):
    # 0 = adelantado (y < -1), 1 = puntual (-1 <= y <= 5), 2 = retrasado (y > 5)
    ...

# vueltas["y"] = ...
# vueltas["clase"] = ...

#### ✅ Verificación 3.3

In [ ]:
def verificar_3_3(vueltas):
    v = vueltas.dropna(subset=["y"])
    assert len(v) == 4004
    assert abs(v.y.mean() - 2.72) < 0.01 and abs(v.y.std(ddof=0) - 8.22) < 0.01
    dist = (v.clase.value_counts(normalize=True).sort_index() * 100).round(1).tolist()
    assert dist == [17.0, 55.3, 27.7], dist
    print("3.3 correcto ✔  clases (%):", dist)

# verificar_3_3(vueltas)

### 3.4 Limpieza del GPS e integración

1. Eliminar duplicados por (`vehicle_id_id`, `date`, `time`). Son 6 783. **Por qué:** un mismo instante registrado dos veces sesga los agregados.
2. Filtrar velocidades físicamente imposibles. El máximo observado es 107 km/h. El umbral se define y justifica en el EDA.
3. Convertir `time` a segundos y ordenar por vehículo, fecha y hora.
4. Cruzar con `vueltas` por (`vehicle_id_id`, `date`, `lap`). La cobertura esperada es del 99,9 %.
5. **Asignar cada registro GPS a su tramo** usando las horas **reales** de paso: un registro con hora `t` pertenece al tramo *j* si `E[i, j-1] ≤ t < E[i, j]`.

**Atención:** el GPS reporta con menor frecuencia cuando el bus está detenido (cada ~75 s en mediana, frente a ~25 s en marcha). Por eso el tiempo detenido debe calcularse como **duración** (suma de intervalos entre registros), no como número de registros.

In [ ]:
# TODO: implementar
# mon = mon_raw.drop_duplicates(...)
# mon["t_seg"] = ...

### 3.5 (Opcional) Ubicación de los puntos de control

Para el mapa del EDA y la demo. Para cada vuelta y PC, se toma el registro GPS más cercano en el tiempo (±30 s) a la hora real de paso, y se calcula la **mediana** de latitud y longitud por PC.

Referencia (Tabla III del paper): PC1 ≈ (−13,5552; −71,8568), PC10 ≈ (−13,5543; −71,9908) y PC13 ≈ (−13,5226; −71,9818), en las inmediaciones del mercado San Pedro.

In [ ]:
# TODO (opcional)

## 4. Análisis exploratorio (OE2)

Gráficos sugeridos para el paper. El Entregable 1 ya incluye versiones preliminares que pueden mejorarse:

1. Distribución de `y`: histograma con líneas en −1 y +5.
2. Retraso medio y % de retraso por **horario programado**. Compárenlo con la Tabla V.
3. Retraso medio por **hora de salida** (Fig. 2).
4. Retraso medio por **día de la semana**.
5. **Cambio medio del retraso por tramo** (Fig. 3). Referencia: PC13→PC14 ≈ +2,79 min y PC14→PC15 ≈ −1,96 min.
6. Mapa de la ruta con los PC.
7. Correlación entre el retraso en PC *k* y el retraso final, para k = 1…17. **Por qué:** muestra cuánta información aporta cada punto de control y motiva la elección de K_VALUES.

In [ ]:
# TODO: gráficos del EDA

## 5. Ingeniería de variables (OE4)

Tres conjuntos de variables (Sección IV-G del paper):

| Conjunto | Contenido | Escenario |
|---|---|---|
| **F0** | hora programada de salida, día de la semana, número de vuelta, duración programada y vehículo | k = 0 y base de todos |
| **F1** | F0 + desviaciones `D[:, 0:k]` | k = 4, 8, 12 |
| **F2** | F1 + agregados GPS de los tramos ya recorridos: velocidad media ponderada por tiempo, desviación de la velocidad y minutos detenido | k = 4, 8, 12 (ablación H3) |

**Regla anti-fuga (crítica):** toda variable para el escenario *k* debe usar **solo información disponible al pasar por PC k**. Para el GPS, eso significa usar solo registros con `t ≤ E[i, k-1]`. Si se usa información posterior, el modelo "ve el futuro" y los resultados no son válidos.

**Faltantes en `D[:, 0:k]`:**
- XGBoost acepta `NaN` de forma nativa.
- Ridge, Logística y Random Forest requieren imputación: último valor observado (*forward fill* a lo largo de los PC) más una columna indicadora de faltante.

**Codificación:**
- Modelos lineales: one-hot para día, vuelta, horario y vehículo; estandarizar las numéricas.
- Modelos de árboles: no necesitan escalado.

In [ ]:
# TODO: implementar
def construir_features(vueltas, D, k, conjunto="F1", gps=None, para_lineal=False):
    # Devuelve la matriz X (DataFrame) para el escenario k.
    #   conjunto: "F0", "F1" o "F2"
    #   gps: tabla de registros GPS ya asignados a tramos (solo para F2)
    #   para_lineal: si True, aplica one-hot e imputación con indicador
    ...

## 6. Partición temporal

- **Prueba:** desde `TEST_INICIO` (5 días, 758 vueltas con `y` válido). **No se toca hasta la etapa 9.**
- **Desarrollo:** 22 días (3 246 vueltas). Sobre él se aplican los 4 pliegues de `CV_FOLDS`.

**Por qué temporal y no aleatoria:** las vueltas de un mismo día comparten condiciones (tráfico, eventos). Una partición aleatoria pondría vueltas "hermanas" en entrenamiento y validación, y sobreestimaría el desempeño. Además, en la operación real siempre se predice el futuro con datos del pasado.

In [ ]:
# TODO: implementar
# mascara_test = ...
# mascara_dev  = ...
def indices_fold(vueltas, fin_tr, ini_va, fin_va):
    # Devuelve (idx_train, idx_val) para un pliegue de CV_FOLDS
    ...

#### ✅ Verificación 6

In [ ]:
def verificar_6(vueltas, mascara_test, mascara_dev):
    v = vueltas["y"].notna()
    assert int((v & mascara_test).sum()) == 758
    assert int((v & mascara_dev).sum()) == 3246
    print("6 correcto ✔")

# verificar_6(vueltas, mascara_test, mascara_dev)

## 7. Líneas base (H1)

| Línea base | Regresión | Clasificación |
|---|---|---|
| **B1. Persistencia** (solo k > 0) | `ŷ = último retraso observado en PC1..PCk` | clase de ese valor |
| **B2. Histórica** | promedio de `y` en entrenamiento por hora de salida | clase mayoritaria |

**Por qué:** un modelo de ML solo aporta valor si supera estas reglas simples (Taparia [9]; Liu *et al.* [6]).

Referencia preliminar (entrenamiento 06–23/01, validación 24–27/01):
- B2 da MAE ≈ 5,36 min; la media global, ≈ 6,69 min; la clase mayoritaria, F1-macro ≈ 0,22.
- Persistencia: k=4 → 7,05; k=8 → 5,04; k=12 → 4,41.

In [ ]:
# TODO: implementar las líneas base y evaluarlas en los 4 pliegues

## 8. Modelos y ajuste de hiperparámetros (OE3)

| Modelo | Regresión | Clasificación | Antecedentes |
|---|---|---|---|
| M1 | `Ridge` | `LogisticRegression` (multinomial) | [9], [11] |
| M2 | `RandomForestRegressor` | `RandomForestClassifier` | [7], [10] |
| M3 | `XGBRegressor` | `XGBClassifier` | [7], [8], [9] |

**Procedimiento por modelo, tarea y escenario (k, conjunto):**
1. Para cada combinación de la malla (`GRIDS_REG` / `GRIDS_CLF`), entrenar y evaluar en los 4 pliegues.
2. Elegir la combinación con **menor MAE medio** (regresión) o **mayor F1-macro medio** (clasificación).
3. Guardar media ± desviación estándar entre pliegues. El paper las reporta así.

**Sugerencia:** `sklearn.model_selection.GridSearchCV` acepta pliegues propios con `cv=[(idx_tr, idx_va), ...]`. Así pueden reutilizar `indices_fold`.

**Ojo con el tiempo de cómputo:** la malla de XGBoost tiene 36 combinaciones × 4 pliegues × 4 escenarios × 2 tareas. Si es lenta, usen `RandomizedSearchCV` con `n_iter` fijo y justifíquenlo en el paper.

Referencia preliminar sin ajuste (validación 24–27/01):

| k | Ridge | RF | XGB | F1 Log. | F1 RF | F1 XGB |
|---|---|---|---|---|---|---|
| 0 | 4,53 | 3,92 | 3,86 | 0,67 | 0,71 | 0,69 |
| 4 | 4,39 | 3,83 | 3,73 | 0,69 | 0,72 | 0,72 |
| 8 | 3,97 | 3,60 | 3,62 | 0,71 | 0,72 | 0,74 |
| 12 | 3,23 | 3,33 | 3,14 | 0,73 | 0,74 | 0,75 |

In [ ]:
# TODO: búsqueda de hiperparámetros con validación cruzada temporal
resultados_cv = []   # lista de dicts: {modelo, tarea, k, conjunto, params, metrica_media, metrica_std}

## 9. Evaluación final en el conjunto de prueba (OE3, H1)

Entrenar cada modelo con sus **mejores hiperparámetros sobre todo el conjunto de desarrollo** y evaluarlo **una sola vez** en prueba.

**Métricas:**
- Regresión: MAE (principal), RMSE y R².
- Clasificación: F1-macro (principal), *recall* de "retrasado", matriz de confusión y PR-AUC uno contra el resto (`average_precision_score` con `predict_proba`).

**Pruebas estadísticas (α = 0,05):**
- Regresión: Wilcoxon de rangos con signo sobre los **errores absolutos pareados por vuelta** del mejor modelo frente a cada línea base: `wilcoxon(err_modelo, err_base)`.
- Clasificación: McNemar sobre los aciertos pareados. Usen la tabla 2×2 de (acierta modelo, acierta base) y `statsmodels.stats.contingency_tables.mcnemar`.

**Por qué pruebas pareadas:** ambos modelos predicen las **mismas** vueltas, así que se compara la diferencia vuelta a vuelta, no solo los promedios.

In [ ]:
# TODO: evaluación final y pruebas estadísticas

## 10. Ablaciones (OE4, H2, H3)

1. **Efecto de k (H2):** graficar la métrica principal frente a k = 0, 4, 8, 12, por modelo. **Pregunta:** ¿en qué punto de la vuelta la información en ruta empieza a aportar? El preliminar sugiere que el aporte es moderado hasta k = 12.
2. **Aporte del GPS (H3):** comparar F1 frente a F2 en cada k > 0. En una exploración inicial, agregados GPS simples **no** redujeron el error. Si se confirma, reportarlo: un resultado nulo bien medido también es un hallazgo.

In [ ]:
# TODO: tablas y gráficos de ablación

## 11. Interpretación (OE5)

Importancia por permutación (`sklearn.inspection.permutation_importance`) del mejor modelo, sobre el conjunto de prueba, con `n_repeats ≥ 10`.

**Pregunta para la discusión:** ¿qué explica la impuntualidad: el contexto del itinerario (hora, día, horario) o lo ocurrido en ruta? En el análisis preliminar, la hora programada fue la variable dominante. Relacionen el resultado con las observaciones de campo: esperas en paraderos concurridos según la demanda y congestión en San Pedro y en los accesos a la Av. de la Cultura.

*Opcional:* SHAP (`!pip install shap`) para explicar predicciones individuales.

In [ ]:
# TODO: importancia de variables

## 12. Reproducibilidad y exportación

- Guardar las tablas de resultados en CSV (`resultados_cv.csv`, `resultados_test.csv`) y las figuras en PNG a 300 dpi para el paper.
- Registrar las versiones de las librerías (celda de la etapa 0) en el paper (Sección V-F).
- Subir el notebook a GitHub **sin** los CSV de datos, y documentar en el README cómo obtenerlos.

In [ ]:
# TODO: exportar resultados
# pd.DataFrame(resultados_cv).to_csv("resultados_cv.csv", index=False)